In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from dotenv import dotenv_values
from IPython.display import clear_output
from omegaconf import OmegaConf
from pathlib import Path

from annotate import display_fixed_image
from beyond_hate.analysis.utils import compute_pairwise_agreement

cfg = OmegaConf.load("../../config/default.yaml")

env_values = dotenv_values()
hf_token = env_values["HF_TOKEN"]

# Resolve paths
project_root = Path("../..").resolve()
calibration_labels_file = project_root / cfg.data.paths.calibration_file
hf_data = project_root / cfg.data.paths.hf

In [ ]:
annotations = pd.read_json(calibration_labels_file, lines=True,
                           dtype={"label_hateful": int,
                                  "label_incivility": str,
                                  "label_intolerance": str})

# Add full path to the image
annotations['img_path'] = annotations['img'].apply(lambda x: hf_data / x)

annotations.loc[:, "label_incivility_bin"]  = annotations["label_incivility"].apply(lambda x: 1 if "0" not in x else 0)
annotations.loc[:, "label_intolerance_bin"] = annotations["label_intolerance"].apply(lambda x: 1 if "0" not in x else 0)

In [ ]:
print(f"Annotators: {list(annotations['annotator'].unique())}")

print("\nINCIVILITY LABELS:")
inciv_agreements, inciv_kappas = compute_pairwise_agreement(annotations, "label_incivility_bin")
if inciv_agreements:
    print(f"Average agreement: {np.mean(inciv_agreements):.3f} ± {np.std(inciv_agreements):.3f}")
    print(f"Average Kappa: {np.mean(inciv_kappas):.3f} ± {np.std(inciv_kappas):.3f}")

# Intolerance labels
print("\nINTOLERANCE LABELS:")
intol_agreements, intol_kappas = compute_pairwise_agreement(annotations, "label_intolerance_bin")
if intol_agreements:
    print(f"Average agreement: {np.mean(intol_agreements):.3f} ± {np.std(intol_agreements):.3f}")
    print(f"Average Kappa: {np.mean(intol_kappas):.3f} ± {np.std(intol_kappas):.3f}")

print("\n" + "=" * 50)

In [ ]:
# Find images with disagreement (across both labels and all annotators)
disagreement_mask = (
    annotations.groupby("id")[["label_incivility_bin", "label_intolerance_bin"]]
    .apply(lambda x: (x["label_incivility_bin"].nunique() > 1) or (x["label_intolerance_bin"].nunique() > 1))
)

disagreed_images = annotations[annotations["id"].isin(disagreement_mask[disagreement_mask].index)]

In [ ]:
map_incivility = {0: "Civil", 1: "Vulgar", 2: "Attacks", 3: "Aspersions"}
map_intolerance = {
    0: "Tolerant",
    1: "Threats to rights",
    2: "Political",
    3: "Racism",
    4: "Social",
    5: "Gender",
    6: "Religious",
    7: "Stereotyping",
    8: "Violent threats",
    9: "Ableism"
}

def map_labels(label_str, label_map):
    """Map comma-separated label indices to their descriptions"""
    indices = [int(x.strip()) for x in label_str.split(",")]
    return ", ".join([label_map[idx] for idx in indices])


In [ ]:
for img_path in disagreed_images["img_path"].unique():
    clear_output(wait=True)
    # Display annotations for the current image
    img_annotations = disagreed_images[disagreed_images["img_path"] == img_path]
    print(f"{'Annotator':<15} | {'Incivility':<40} | {'Intolerance':<40}")
    print("-" * 100)
    for row in img_annotations.itertuples():
        incivility_mapped = map_labels(row.label_incivility, map_incivility)
        intolerance_mapped = map_labels(row.label_intolerance, map_intolerance)
        print(f"{row.annotator:<15} | {incivility_mapped:<40} | {intolerance_mapped:<40}")
    # Display the image
    display_fixed_image(img_path, size=(400, 400))

    user_input = input()
    if user_input.lower() == 'q':
        break